# 🎬 Antigravity 4K Motion Graphics Batch Renderer (Real WebGL Canvas Engine)
Studio Otomatis Produksi Video Loop 4K (3840x2160, 30 FPS, Seamless Loop) untuk Adobe Stock & Freepik.

### ⚡ Cara Kerja Engine:
1. **Meng-clone Repo Web Previewer Langsung dari GitHub** (`consistmaker/shadergradientpaper`).
2. **Menjalankan Puppeteer Headless Chrome dengan Akses WebGL GPU Penuh**.
3. **Menginjeksi Konfigurasi JSON Anda** ke dalam kanvas WebGL asli (Paper Shaders & ShaderGradient).
4. **Merekam Setiap Frame 4K Asli** dan meng-encode menjadi video MP4 4K 30fps Ultra High Quality via Hardware FFmpeg.
5. **Mengemas Seluruh Video ke ZIP Siap Jual**.

## ⚙️ Step 1: Clone Web App & Install Puppeteer WebGL GPU Renderer

In [ ]:
# 1. Install System FFmpeg dan Google Chrome GPU Driver
!apt-get update -qq
!apt-get install -y ffmpeg chromium-browser libgbm-dev libnss3 libasound2 > /dev/null 2>&1

# 2. Clone Repository Web App Antigravity Studio dari GitHub
!rm -rf /content/shadergradientpaper
!git clone https://github.com/consistmaker/shadergradientpaper.git /content/shadergradientpaper

# 3. Install NPM Dependencies & Build Production WebGL Studio
%cd /content/shadergradientpaper
!npm install --legacy-peer-deps > /dev/null 2>&1
!npm install puppeteer-core > /dev/null 2>&1
!npm run build

print("\n✅ REPOSITORY CLONED & 4K WEBGL HEADLESS ENGINE READY!")

## 📥 Step 2: Masukkan Recipe JSON dari Web Live Previewer
Paste JSON hasil tombol **Export Batch** dari web Live Previewer di bawah ini.

In [ ]:
import json
import os

# Paste JSON dari Live Previewer di sini:
RECIPE_JSON = '''
{
  "metadata": {
    "targetResolution": "3840x2160 (4K UHD)",
    "targetFps": 30,
    "loopDurationSeconds": 10,
    "isSeamlessLoop": true,
    "batchMode": "manual_queue"
  },
  "manualQueueList": [
    {
      "index": 1,
      "id": "item_1",
      "name": "Paper: mesh-gradient (#e0eaff)",
      "engine": "paper",
      "config": {
        "shaderType": "mesh-gradient",
        "color1": "#e0eaff",
        "color2": "#241d9a",
        "color3": "#f75092",
        "color4": "#9f50d3",
        "speed": 1.0,
        "distortion": 0.8,
        "swirl": 0.1
      }
    }
  ],
  "totalVideosInQueue": 1
}
'''

with open('/content/render_recipe.json', 'w') as f:
    f.write(RECIPE_JSON.strip())

recipe = json.loads(RECIPE_JSON)
batch_mode = recipe['metadata'].get('batchMode', 'manual_queue')
print(f"🎯 Batch Mode: {batch_mode.upper()}")
print(f"🎬 Target Specs: {recipe['metadata']['targetResolution']} @ {recipe['metadata']['targetFps']} FPS ({recipe['metadata']['loopDurationSeconds']}s Loop)")
print("✅ Recipe JSON Saved to /content/render_recipe.json")

## 🎲 Step 3: Siapkan Script Headless WebGL Capture Engine (Node.js Puppeteer)

In [ ]:
renderer_script = '''
const puppeteer = require('puppeteer-core');
const fs = require('fs');
const path = require('path');
const { spawn } = require('child_process');
const http = require('http');

// Simple static server for dist/
const distDir = path.join('/content/shadergradientpaper/dist');
const server = http.createServer((req, res) => {
  let filePath = path.join(distDir, req.url === '/' ? 'index.html' : req.url);
  if (!fs.existsSync(filePath)) filePath = path.join(distDir, 'index.html');
  
  const ext = path.extname(filePath);
  let contentType = 'text/html';
  if (ext === '.js') contentType = 'text/javascript';
  else if (ext === '.css') contentType = 'text/css';
  else if (ext === '.svg') contentType = 'image/svg+xml';
  
  res.writeHead(200, { 'Content-Type': contentType });
  fs.createReadStream(filePath).pipe(res);
});

server.listen(4173, async () => {
  console.log('📡 Local Headless WebGL Server live on port 4173');
  
  const recipe = JSON.parse(fs.readFileSync('/content/render_recipe.json', 'utf8'));
  const outputDir = '/content/output_4k_videos';
  if (!fs.existsSync(outputDir)) fs.mkdirSync(outputDir, { recursive: true });

  const browser = await puppeteer.launch({
    executablePath: '/usr/bin/chromium-browser',
    headless: 'new',
    args: [
      '--no-sandbox',
      '--disable-setuid-sandbox',
      '--use-gl=angle',
      '--use-angle=gl',
      '--enable-webgl',
      '--ignore-gpu-blocklist',
      '--window-size=3840,2160'
    ]
  });

  const queue = recipe.manualQueueList || [
    {
      id: 'single',
      name: 'Video',
      engine: recipe.metadata?.engine || 'paper',
      config: recipe.baseConfig || {}
    }
  ];

  console.log(`🎬 Total Videos to Render: ${queue.length}`);

  for (let i = 0; i < queue.length; i++) {
    const item = queue[i];
    const outputFile = path.join(outputDir, `motion_4k_${item.engine}_${i + 1}.mp4`);
    console.log(`\n🎥 [${i + 1}/${queue.length}] Rendering Real WebGL 4K: ${item.name}...`);

    const page = await browser.newPage();
    await page.setViewport({ width: 3840, height: 2160, deviceScaleFactor: 1 });
    await page.goto('http://localhost:4173/', { waitUntil: 'networkidle0' });

    // Inject current shader configuration
    await page.evaluate((conf, eng) => {
      window.__SET_ENGINE_RENDER?.(eng, conf);
    }, item.config, item.engine);

    await new Promise(r => setTimeout(r, 2000)); // Allow WebGL shader to compile

    // Start FFmpeg hardware stream from raw canvas frames
    const fps = recipe.metadata?.targetFps || 30;
    const duration = recipe.metadata?.loopDurationSeconds || 10;
    const totalFrames = fps * duration;

    const ffmpeg = spawn('ffmpeg', [
      '-y',
      '-f', 'image2pipe',
      '-vcodec', 'png',
      '-r', `${fps}`,
      '-i', '-',
      '-c:v', 'libx264',
      '-pix_fmt', 'yuv420p',
      '-b:v', '45M',
      '-preset', 'fast',
      outputFile
    ]);

    console.log(`   ⏳ Capturing ${totalFrames} frames in 4K resolution (3840x2160)...`);
    for (let f = 0; f < totalFrames; f++) {
      const screenshot = await page.screenshot({ type: 'png' });
      ffmpeg.stdin.write(screenshot);
      if (f % 60 === 0) console.log(`      Frame ${f}/${totalFrames} encoded...`);
    }

    ffmpeg.stdin.end();
    await new Promise(resolve => ffmpeg.on('close', resolve));
    await page.close();
    console.log(`   ✅ Success 4K Render: ${outputFile}`);
  }

  await browser.close();
  server.close();
  console.log('\n🎉 ALL REAL WEBGL 4K VIDEOS HAVE BEEN RENDERED SUCCESSFULLY!');
  process.exit(0);
});
'''

with open('/content/headless_renderer.js', 'w') as f:
    f.write(renderer_script.strip())

print("✅ WebGL Puppeteer Capture Engine Ready!")

## 🚀 Step 4: Eksekusi Render 4K WebGL (Sesuai Persis dengan Previewer)

In [ ]:
# Jalankan Headless WebGL Capture
!node /content/headless_renderer.js

## 📦 Step 5: Download Seluruh Video 4K sebagai ZIP

In [ ]:
from google.colab import files
!zip -j -r /content/4K_Motion_Graphics_Batch.zip /content/output_4k_videos
print("\n📦 Packaging Complete! Downloading ZIP file...")
files.download('/content/4K_Motion_Graphics_Batch.zip')